# 06 · Sequential testing (mSPRT) — `foe.sequential`

**Hexkit surface:** `pages/sequential_analysis.py` (`calculate_boundaries`, `process_test_trajectory`).

FOE uses the closed-form mixture SPRT statistic (Johari et al.): with V the variance of the
observed difference d and a N(0, τ) mixing prior on the true effect θ,

Λ = ∫ φ(d; θ, V) / φ(d; 0, V) · φ(θ; 0, τ) dθ,  log Λ = ½ [ log(V / (V + τ)) + d²/V · τ / (V + τ) ]

R evaluates the **integral itself** with `integrate`, so the closed form, the pooled/one-sample
variance choice and the trajectory bookkeeping are all checked against the definition.

| FOE function | R reference |
|---|---|
| `calculate_llr_vectorized` (multi-sample pooled, one-sample) | `log(integrate(...))` of the mixture likelihood ratio |
| `calculate_boundaries` | `log(k / α)`, `log(β)` |
| `calculate_instantaneous_power_vectorized` | `pwr::pwr.norm.test` power function (one-sided) |
| `process_test_trajectory` — per-day LLR and winner / loser / continue status | R recomputes the LLR per day from cumulative counts and applies the boundaries |

In [1]:
import os
os.environ.setdefault("RPY2_CFFI_MODE", "ABI")  # use rpy2 ABI mode; avoids a noisy API-mode probe
%load_ext rpy2.ipython

import numpy as np
import pandas as pd

from foe.sequential.operations import SequentialEngine, TestType
from validation_helpers import Checks

checks = Checks()
eng = SequentialEngine()

In [2]:
%%R
# log of the mixture likelihood ratio, integrated numerically
mixture_llr <- function(d, V, tau) {
  log_f <- function(theta) dnorm(d, theta, sqrt(V), log = TRUE) - dnorm(d, 0, sqrt(V), log = TRUE) +
    dnorm(theta, 0, sqrt(tau), log = TRUE)
  # The integrand has its mass around `centre`; rescale by its value there to avoid overflow.
  centre <- d * tau / (V + tau); spread <- sqrt(V * tau / (V + tau))
  shift <- log_f(centre)
  val <- integrate(function(t) exp(log_f(t) - shift), centre - 40 * spread, centre + 40 * spread,
                   rel.tol = 1e-10)$value
  log(val) + shift
}
llr_two_sample <- function(n_c, x_c, n_v, x_v, tau) {
  p_pool <- (x_c + x_v) / (n_c + n_v)
  V <- p_pool * (1 - p_pool) * (1 / n_c + 1 / n_v)
  mixture_llr(x_v / n_v - x_c / n_c, V, tau)
}
llr_one_sample <- function(n, x, p0, tau) mixture_llr(x / n - p0, p0 * (1 - p0) / n, tau)

## 1. LLR at single points

In [3]:
pts = pd.DataFrame([
    # n_c, x_c, n_v, x_v, tau
    (1_000,   30,  1_000,   45, 0.02 ** 2),
    (20_000, 600, 20_000,  690, 0.01 ** 2),
    (50_000, 2_500, 50_100, 2_480, 0.005 ** 2),
    (5_000,  900,  5_050,  1_010, 0.03 ** 2),
    (300_000, 9_000, 300_000, 9_350, 0.002 ** 2),
], columns=["n_c", "x_c", "n_v", "x_v", "tau"])
pts["llr_two_py"] = [float(eng.calculate_llr_vectorized(np.array([r.n_v]), np.array([r.x_v]),
                                                        np.array([r.n_c]), np.array([r.x_c]), tau=r.tau)[0])
                     for r in pts.itertuples()]
p0 = 0.03
pts["llr_one_py"] = [float(eng.calculate_llr_vectorized(np.array([r.n_v]), np.array([r.x_v]),
                                                        tau=r.tau, fixed_baseline_cr=p0)[0])
                     for r in pts.itertuples()]
pts

,n_c,x_c,n_v,x_v,tau,llr_two_py,llr_one_py
0,1000,30,1000,45,0.000400,0.381134,2.258327
1,20000,600,20000,690,0.000100,1.397098,4.736660
2,50000,2500,50100,2480,0.000025,-1.266531,318.036469
3,5000,900,5050,1010,0.000900,1.679812,2489.163923
4,300000,9000,300000,9350,0.000004,1.753168,4.978276


In [4]:
%%R -i pts -i p0 -o llr_r
llr_r <- data.frame(
  two = sapply(seq_len(nrow(pts)), function(i) with(pts[i, ], llr_two_sample(n_c, x_c, n_v, x_v, tau))),
  one = sapply(seq_len(nrow(pts)), function(i) with(pts[i, ], llr_one_sample(n_v, x_v, p0, tau)))
)
llr_r

        two

         one


1

  0.3811341

    2.258327


2

... (7 outputs left out)

 2489.163923


5

  1.7531677

    4.978276

In [5]:
for i, r in pts.iterrows():
    checks.add(f"LLR two-sample n_v={r.n_v} x_v={r.x_v}", r.llr_two_py, llr_r.two.iloc[i], atol=1e-8, rtol=1e-8)
    checks.add(f"LLR one-sample n_v={r.n_v} x_v={r.x_v}", r.llr_one_py, llr_r.one.iloc[i], atol=1e-8, rtol=1e-8)

## 2. Boundaries and instantaneous power

In [6]:
bounds_py = [eng.calculate_boundaries(a, b, k) for a, b, k in [(0.05, 0.2, 1), (0.05, 0.2, 3), (0.01, 0.1, 2)]]
n_var = np.array([2_000, 10_000, 40_000]); n_ctrl = np.array([2_000, 10_100, 39_900])
p0_arr = np.array([0.03, 0.05, 0.10]); mde, alpha = 0.01, 0.05
power_py = eng.calculate_instantaneous_power_vectorized(n_var, p0_arr, mde, alpha, n_ctrl=n_ctrl)
power_one_py = eng.calculate_instantaneous_power_vectorized(n_var, 0.04, mde, alpha)

In [7]:
%%R -i n_var -i n_ctrl -i p0_arr -i mde -i alpha -o other_r
suppressPackageStartupMessages(library(pwr))
bounds <- c(log(1 / 0.05), log(0.2), log(3 / 0.05), log(0.2), log(2 / 0.01), log(0.1))
# one-sided z-test power: pnorm(mde / se - z_alpha) == pwr.norm.test(d = mde / se, n = 1, "greater")
p1 <- p0_arr + mde
se2 <- sqrt(p0_arr * (1 - p0_arr) / n_ctrl + p1 * (1 - p1) / n_var)
pw2 <- sapply(seq_along(se2), function(i) pwr.norm.test(d = mde / se2[i], n = 1, sig.level = alpha, alternative = "greater")$power)
se1 <- sqrt((0.04 + mde) * (1 - 0.04 - mde) / n_var)
pw1 <- sapply(seq_along(se1), function(i) pwr.norm.test(d = mde / se1[i], n = 1, sig.level = alpha, alternative = "greater")$power)
other_r <- c(bounds, pw2, pw1)   # 6 boundaries, 3 two-sample, 3 one-sample

In [8]:
for j, (up, lo) in enumerate(bounds_py):
    checks.add(f"upper boundary case {j+1}", up, other_r[2 * j], rtol=1e-12)
    checks.add(f"lower boundary case {j+1}", lo, other_r[2 * j + 1], rtol=1e-12)
for i in range(len(n_var)):
    checks.add(f"power two-sample n={n_var[i]}", power_py[i], other_r[6 + i], rtol=1e-10)
    checks.add(f"power one-sample n={n_var[i]}", power_one_py[i], other_r[9 + i], rtol=1e-10)

## 3. Full trajectory: A/B/C test over 28 days

Daily data with staggered challenger start, converted to cumulative counts as the page does.
R recomputes every day's LLR and status independently.

In [9]:
rng = np.random.default_rng(42)
dates = pd.date_range("2026-03-01", periods=28)
rates = {"Control": 0.040, "Variant B": 0.047, "Variant C": 0.039}
rows = []
for v, p in rates.items():
    start = 3 if v == "Variant C" else 0               # Variant C starts on day 4
    daily_n = rng.poisson(1_500, len(dates))
    daily_x = rng.binomial(daily_n, p)
    daily_n[:start] = 0; daily_x[:start] = 0
    for d, n, x in zip(dates[start:], np.cumsum(daily_n)[start:], np.cumsum(daily_x)[start:]):
        rows.append(dict(measurement_date=d, variant_name=v, visitors=int(n), conversions=int(x)))
cum = pd.DataFrame(rows)

tau, a, b = 0.008 ** 2, 0.05, 0.2
traj = eng.process_test_trajectory(cum, TestType.MULTI_SAMPLE, tau=tau, alpha=a, beta=b,
                                   num_variants=2, control_group_name="Control")
traj_py = traj[["measurement_date", "variant_name", "visitors_ctrl", "conversions_ctrl",
                "visitors_var", "conversions_var", "llr", "status"]].reset_index(drop=True)
traj_py["date"] = traj_py.measurement_date.dt.strftime("%Y-%m-%d")
traj_py.drop(columns="measurement_date").tail()

,variant_name,visitors_ctrl,conversions_ctrl,visitors_var,conversions_var,llr,status,date
51,Variant C,36076,1409,31380.0,1267.0,-1.316559,continue,2026-03-24
52,Variant C,37600,1477,32865.0,1317.0,-1.569058,continue,2026-03-25
53,Variant C,39095,1531,34351.0,1369.0,-1.618715,loser,2026-03-26
54,Variant C,40547,1583,35828.0,1437.0,-1.472497,continue,2026-03-27
55,Variant C,42066,1638,37393.0,1505.0,-1.334613,continue,2026-03-28


In [10]:
cum_r = cum.assign(date=cum.measurement_date.dt.strftime("%Y-%m-%d")).drop(columns="measurement_date")

In [11]:
%%R -i cum_r -i tau -i a -i b -o traj_r
upper <- log(2 / a); lower <- log(b)
ctrl <- cum_r[cum_r$variant_name == "Control", ]
traj_r <- do.call(rbind, lapply(setdiff(unique(cum_r$variant_name), "Control"), function(v) {
  var <- cum_r[cum_r$variant_name == v, ]
  m <- merge(var, ctrl, by = "date", suffixes = c("_var", "_ctrl"))
  m$llr <- mapply(llr_two_sample, m$visitors_ctrl, m$conversions_ctrl, m$visitors_var, m$conversions_var,
                  MoreArgs = list(tau = tau))
  m$status <- ifelse(m$llr >= upper, "winner", ifelse(m$llr <= lower, "loser", "continue"))
  data.frame(date = m$date, variant_name = v, llr = m$llr, status = m$status)
}))
tail(traj_r)

       date

 variant_name

       llr

   status


48

 2026-03-23

... (24 outputs left out)

 2026-03-28

    Variant C

 -1.334613

 continue

In [12]:
# FOE outer-joins the dates; before Variant C starts its rows are filled with 0 and get LLR 0.
merged = traj_py.merge(traj_r, on=["date", "variant_name"], how="left", suffixes=("_py", "_r"))
pre_start = merged.llr_r.isna()
for _, r in merged[pre_start].iterrows():
    checks.add(f"pre-start LLR is 0 {r.variant_name} {r.date}", r.llr_py, 0.0)
for _, r in merged[~pre_start].iterrows():
    checks.add(f"LLR    {r.variant_name} {r.date}", r.llr_py, r.llr_r, atol=1e-8, rtol=1e-8)
    checks.add_equal(f"status {r.variant_name} {r.date}", r.status_py, r.status_r)
merged.groupby("variant_name").tail(1)[["variant_name", "date", "llr_py", "llr_r", "status_py", "status_r"]]

,variant_name,date,llr_py,llr_r,status_py,status_r
27,Variant B,2026-03-28,14.235239,14.235239,winner,winner
55,Variant C,2026-03-28,-1.334613,-1.334613,continue,continue


## Summary

In [13]:
checks.report()

131 checks, 131 passed, 0 failed

Largest differences:
                                check  hexkit       R    abs_diff              tolerance
 LLR one-sample n_v=5050.0 x_v=1010.0 2489.16 2489.16 9.09495e-13 atol=1e-08, rtol=1e-08
LLR one-sample n_v=50100.0 x_v=2480.0 318.036 318.036 5.68434e-14 atol=1e-08, rtol=1e-08
          LLR    Variant B 2026-03-23 10.7816 10.7816 7.10543e-15 atol=1e-08, rtol=1e-08
          LLR    Variant B 2026-03-18 8.53513 8.53513 3.55271e-15 atol=1e-08, rtol=1e-08
          LLR    Variant B 2026-03-27 12.6257 12.6257 3.55271e-15 atol=1e-08, rtol=1e-08


,check,hexkit,R,abs_diff,tolerance,pass
0,LLR two-sample n_v=1000.0 x_v=45.0,0.381134,0.381134,2.220446e-16,"atol=1e-08, rtol=1e-08",True
1,LLR one-sample n_v=1000.0 x_v=45.0,2.258327,2.258327,4.440892e-16,"atol=1e-08, rtol=1e-08",True
2,LLR two-sample n_v=20000.0 x_v=690.0,1.397098,1.397098,2.220446e-16,"atol=1e-08, rtol=1e-08",True
3,LLR one-sample n_v=20000.0 x_v=690.0,4.73666,4.73666,0.000000e+00,"atol=1e-08, rtol=1e-08",True
4,LLR two-sample n_v=50100.0 x_v=2480.0,-1.266531,-1.266531,4.440892e-16,"atol=1e-08, rtol=1e-08",True
...,...,...,...,...,...,...
126,status Variant C 2026-03-26,loser,loser,NaN,exact,True
127,LLR Variant C 2026-03-27,-1.472497,-1.472497,4.440892e-16,"atol=1e-08, rtol=1e-08",True
128,status Variant C 2026-03-27,continue,continue,NaN,exact,True
129,LLR Variant C 2026-03-28,-1.334613,-1.334613,6.661338e-16,"atol=1e-08, rtol=1e-08",True
